# FLEO-FER — clean Kaggle run (persist-safe, quota-friendly)

**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **ON**.
**Add Input:** attach `msambare/fer2013` **and** `shuvoalok/raf-db-dataset`.

**Don't lose your work:** use **Save Version -> Save & Run All (Commit)** — it runs top-to-bottom and *saves all output permanently*. 60 epochs fit inside the 12h limit.

Every training zips its outputs right after finishing, so even interactively you can download immediately.

FLEO now trains with dropout + stronger orthogonality + cosine LR (anti-overfit) automatically.

## 1. Clone + install

In [ ]:
%cd /kaggle/working
!rm -rf FLEO
!git clone https://github.com/olfa-askri/FLEO.git
%cd /kaggle/working/FLEO
!pip install -q ultralytics onnx onnxruntime onnxscript
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Auto-find the datasets + prepare (works whatever the mount name)

In [ ]:
import glob, os
def find_root(keys):
    for p in sorted(glob.glob('/kaggle/input/*'))+sorted(glob.glob('/kaggle/input/*/*'))+sorted(glob.glob('/kaggle/input/*/*/*')):
        if os.path.isdir(p) and any(k in p.lower() for k in keys):
            return p
    return None
fer = find_root(['fer2013','fer-2013'])
raf = find_root(['raf-db','rafdb','raf_db'])
print('FER root:', fer); print('RAF root:', raf)
assert fer and raf, 'Attach msambare/fer2013 and shuvoalok/raf-db-dataset via Add Input!'

In [ ]:
!python -m data.prepare_fer2013 --src "{fer}" --out /kaggle/working/FLEO/datasets/fer2013
!python -m data.prepare_rafdb   --src "{raf}" --out /kaggle/working/FLEO/datasets/rafdb
!ls -l /kaggle/working/FLEO/datasets/fer2013/data.yaml /kaggle/working/FLEO/datasets/rafdb/data.yaml

## 3. FER2013 — baseline + FLEO + export R1/R2/R3 + Δ_fold  (60 epochs)

In [ ]:
!python -m scripts.run_matrix --data /kaggle/working/FLEO/datasets/fer2013/data.yaml --dataset fer2013 \n    --seeds 0 --epochs 60 --imgsz 160 --batch 64 --device 0

## 4. FER2013 — Δ_fold, FLEO macro-F1, and baseline accuracy

In [ ]:
import json
print('=== FER2013 Δ_fold + accuracy (full/folded/householder) ===')
print(json.dumps(json.load(open('/kaggle/working/FLEO/results/deltas_fer2013.json')), indent=2))
Wf='/kaggle/working/FLEO/runs/fleo/fleo_seed0/weights/best.pt'
Wb='/kaggle/working/FLEO/runs/fleo/baseline_seed0/weights/best.pt'
print('
=== FLEO macro-F1 ==='); 
!python -m scripts.deltas --data /kaggle/working/FLEO/datasets/fer2013/data.yaml --dataset fer2013 --imgsz 160 --device 0 --weights {Wf} --metric macro_f1
print('
=== BASELINE (accuracy + macro-F1) ==='); 
!python -m scripts.evaluate --data /kaggle/working/FLEO/datasets/fer2013/data.yaml --weights {Wb} --imgsz 160 --device 0

## 5. Zip FER2013 — download NOW from the Output panel

In [ ]:
%cd /kaggle/working/FLEO
!zip -qr /kaggle/working/fleo_fer2013.zip runs export results
!ls -lh /kaggle/working/fleo_fer2013.zip

## 6. RAF-DB — full matrix (60 epochs)

In [ ]:
!python -m scripts.run_matrix --data /kaggle/working/FLEO/datasets/rafdb/data.yaml --dataset rafdb \n    --seeds 0 --epochs 60 --imgsz 160 --batch 64 --device 0

## 7. RAF-DB — the numbers

In [ ]:
import json
print('=== RAF-DB Δ_fold + accuracy ===')
print(json.dumps(json.load(open('/kaggle/working/FLEO/results/deltas_rafdb.json')), indent=2))
Wb='/kaggle/working/FLEO/runs/fleo/baseline_seed0/weights/best.pt'
print('
=== BASELINE ==='); 
!python -m scripts.evaluate --data /kaggle/working/FLEO/datasets/rafdb/data.yaml --weights {Wb} --imgsz 160 --device 0

## 8. Final bundle — download + Save Version

In [ ]:
%cd /kaggle/working/FLEO
!zip -qr /kaggle/working/fleo_all.zip runs export results
!ls -lh /kaggle/working/fleo_all.zip
print('DONE. Download fleo_all.zip from Output, and Save Version to persist.')